**Install & Import Libraries**

In [1]:
# !pip install pandas tabula-py
# !java -version
import pandas as pd
import os
import tabula
from tabula.io import read_pdf
import warnings
warnings.filterwarnings("ignore")

In the **first step**, I begin the process by identifying and storing the headers of each table.
1. Define a list of <span style="color:red">persian keywords</span> that will be used to recognize table headers based on the content of the first column.

2. Create a <span style="color:red">dictionary</span> to store the detected headers along with their corresponding row index and page number.

The following image shows an example of how a table header is detected (highlighted in red).

<div align="center">
  <img src="image/1.HEADER.png" width="700"/>
</div>

Set the Directory
- Choose PDF 1: (Roozaneh.pdf)

In [2]:
os.chdir("../3.OUT_PDF")
PDF1 = "1.Roozaneh.pdf"

The following is the list of persian keywords for PDF file (1):

<div align="center">

`["دانشگاه", "آموزشكده", "دانشكده", "مجتمع", "مركز", "ادامه"]`

</div>

In [3]:
keywords1 = ["دانشگاه", "آموزشكده", "دانشكده", "مجتمع", "مركز", "ادامه"]

uni_headers1 = {}

for page in range(1,23):
    content = tabula.read_pdf(PDF1, lattice = False, guess = False,
                                  pandas_options = {'header': None}, pages = str(page))[0]

    for idx, col in content.iterrows():
        
        if page == 22 and idx >= 36:
            break

        if any(str(cell).startswith(tuple(keywords1)) for cell in col):
            # print( "page:", page, "index:", idx, "university name:", col[0])

            uni_name = str(col[0])
            uni_headers1[(page, idx)] = uni_name

uni_headers1 = list(uni_headers1.items())

uni_headers1[:4] # Test

[((1, 7), 'دانشگاه آيت اله بروجردي-  بروجرد'),
 ((1, 19), 'دانشگاه اراك'),
 ((1, 36), 'دانشگاه اروميه'),
 ((1, 62), 'دانشگاه اصفهان')]

In the **second step**, I continue the process by defining a function that determines which university each row of the table belongs to.

The following function takes 3 inputs:
1. Current Page Number
2. Current Row Index
3. List of Header Entries

During the process, the function performs the following checks:
- For headers on the same page, the function checks whether the current row index is <span style="color:red">greater</span> than the header index.
- If this condition is satisfied, the function <span style="color:red">updates</span> the current university name.

In general, this function returns the <span style="color:red">most recent matching university name</span> for the given row.

In [4]:
def find_university(page, idx, header_items):
    current_uni = None
    
    for (h_page, h_idx), h_name in header_items:
        if h_page < page:
            continue

        if h_page > page:
            break

        if idx > h_idx:
            current_uni = h_name
        else:
            break
        
    return current_uni

In the **third step**, I continue the process by identifying and storing the desired rows from each table.
1. Identify the rows that begin with a <span style="color:red">number</span>.
2. Use the `find_university` function to find matching entries.
3. Make necessary adjustments for rows that contain <span style="color:red">errors</span>.
4. Store the resulting rows to create a dataset.

**Define Specific Areas via Tabula**

In this part, Tabula uses a point-based coordinate system:
- **Unit:** points  
  *(1 point = 1/72 inch)*

The following list contains the coordinates of the vertical lines. (from Left to Right)

<div align="center">

`[205, 233, 261, 290, 318, 389, 531]`

</div>

The following image shows an example of how the rows is detected (highlighted in red & blue).

<div align="center">
  <img src="image/2.ROW.png" width="700"/>
</div>

In [5]:
page_list1 = []

for page in range(1,23):
    content = tabula.read_pdf(PDF1, lattice = False,
                               guess = False, columns = [205, 233, 261, 290, 318, 389, 531],
                              pandas_options = {'header': None}, pages = str(page))[0]
    university = ''

    for idx, col in content.iterrows():

        if str(col[7]).isdigit():
            idx_prim = idx - 1 if (8 <= page <= 11 or 13 <= page <= 15 or page == 17 or 19 <= page <= 20) else idx
            university = find_university(page, idx_prim, uni_headers1)
            row_list = list(col.values)
            row_list += [university] 
            row_list.reverse()

            if page == 7 and row_list[1] == "1465":
                row_list[3] = "الکترونیک، بیوالکتریک، قدرت، کنترل، مخابرات"
            if page == 8 and row_list[1] == "1530":
                row_list[8] = "گرايش حالت جامد - محل تحصيل كرج )عدم واگذاري خوابگاه به پسران("
            if page == 10 and row_list[1] == "1627" or row_list[1] == "1633":
                row_list[3] = "الكترونيك, قدرت, كنترل, مخابرات"
            if page == 11 and row_list[1] == "1679":
                row_list[3] == "اتمـــي, حالـــت جامـــد, هسته اي"
            if page == 12 and row_list[1] == "1708" or row_list[1] == "1746":
                row_list[3] = "الكترونيك, قدرت, كنترل, مخابرات"
            if page == 13 and row_list[1] == "1824":
                row_list[3] = "الكترونيك, قدرت, كنترل, مخابرات"
            if page == 14 and row_list[1] == "1849":
                row_list[3] = "الكترونيـــك, بيوالكتريـــك, قــدرت, كنتــرل, مخــابرات, سيستمهاي ديجيتال"
            if page == 15 and row_list[1] == "1893":
                row_list[3] = "الكترونيك, قدرت, كنترل, مخابرات"
            if page == 16 and row_list[1] == "1956":
                row_list[3] = "الكترونيك, قدرت, كنترل, مخابرات"
            if page == 17 and row_list[1] == "2016":
                row_list[8] = "محـل تحـصيل مركزآمـوزش عـالي كوهدشـت - اسـتفاده از غـذاخوري دانشگاه بصورت آزاد و فاقد خوابگاه"
            if page == 17 and row_list[1] == "2025":
                row_list[8] = "محل تحصيل مركز آموزش عالي الشتر- اسـتفاده از غـذاخوري دانـشگاه بصورت آزاد و فاقد خوابگاه"
            if page == 17 and row_list[1] == "2026":
                row_list[8] = "محل تحصيل مركز آموزش عالي پلدختر- استفاده از غـذاخوري دانـشگاه بصورت آزاد و فاقد خوابگاه"
            if page == 17 and row_list[1] == "6877" or row_list[1] == "6878" or row_list[1] == "6879":
                row_list[0] = "دانشگاه لرستان - خرم آباد"
            if page == 17 and row_list[1] == "6884" or row_list[1] == "6885" or row_list[1] == "6886":
                row_list[0] = "دانشگاه مازندران - بابلسر"
            if page == 19 and row_list[1] == "2125":
                row_list[3] = "الكترونيــــك, قــــدرت, مخابرات"
            if page == 19 and row_list[1] == "2139":
                row_list[2] = "کاردانی علمی کاربردی ماشینهای کشاورزی-مکانیزاسیون ماشینهای کشاورزی"
            if page == 22 and str(col[7]) == "2223":
                break
            
            if page == 1 and idx < 20:
                print(idx, row_list) 
            page_list1.append(row_list)

13 ['دانشگاه آيت اله بروجردي-  بروجرد', '1001', 'رياضيات وكاربردها', nan, '40', '-', 'زن', 'مرد', nan]
14 ['دانشگاه آيت اله بروجردي-  بروجرد', '1002', 'فيزيك', nan, '40', '-', 'زن', 'مرد', nan]
15 ['دانشگاه آيت اله بروجردي-  بروجرد', '1003', 'مهندسي عمران', nan, '40', '-', 'زن', 'مرد', nan]
16 ['دانشگاه آيت اله بروجردي-  بروجرد', '1004', 'مهندسي كامپيوتر', 'نرم افزار', '40', '-', 'زن', 'مرد', nan]
17 ['دانشگاه آيت اله بروجردي-  بروجرد', '6701', 'علوم اقتصادي', 'اقتصادبازرگاني', '40', '-', 'زن', 'مرد', nan]
18 ['دانشگاه آيت اله بروجردي-  بروجرد', '6702', 'علوم اقتصادي', 'اقتصادنظري', '-', '40', 'زن', 'مرد', nan]


Convert extracted tables into a DataFrame with labeled columns

In [6]:
df1 = pd.DataFrame(page_list1, columns = ['University', 'Program-Code', 'Program-Major-Name', 'Field-Type',
                           'First-Cap', 'Second-Cap', 'Female-Cap', 'Male-Cap', 'Description'])

df1.head(5) # Test

,University,Program-Code,Program-Major-Name,Field-Type,First-Cap,Second-Cap,Female-Cap,Male-Cap,Description
0,دانشگاه آيت اله بروجردي- بروجرد,1001,رياضيات وكاربردها,NaN,40,-,زن,مرد,NaN
1,دانشگاه آيت اله بروجردي- بروجرد,1002,فيزيك,NaN,40,-,زن,مرد,NaN
2,دانشگاه آيت اله بروجردي- بروجرد,1003,مهندسي عمران,NaN,40,-,زن,مرد,NaN
3,دانشگاه آيت اله بروجردي- بروجرد,1004,مهندسي كامپيوتر,نرم افزار,40,-,زن,مرد,NaN
4,دانشگاه آيت اله بروجردي- بروجرد,6701,علوم اقتصادي,اقتصادبازرگاني,40,-,زن,مرد,NaN


Save the DataFrame as a CSV file in the **"OUT_TABLE"** folder.

In [7]:
os.chdir("../4.OUT_TABLE")
df1.to_csv("1.Roozaneh.csv", index = False, encoding = "utf-8-sig")

---

Set the Directory
- Choose PDF 2: (Mahroum.pdf)

In [8]:
os.chdir("../3.OUT_PDF")
PDF2 = "2.Mahroum.pdf"

The following is the list of persian keywords for PDF file (2):

<div align="center">

`["مخصوص", "ادامه"]`

</div>

We will use the main pattern again to extract tables from this PDF file.

In [9]:
keywords2 = ["مخصوص", "ادامه"]

uni_headers2 = {}

for page in range(1,3):
    content = tabula.read_pdf(PDF2, lattice = False, guess = False,
                                  pandas_options = {'header': None}, pages = str(page))[0]

    for idx, col in content.iterrows():

        if any(str(cell).startswith(tuple(keywords2)) for cell in col):
            # print( "page:", page, "index:", idx, "university name:", col[0])
            
            uni_name = str(col[0])
            uni_headers2[(page, idx)] = uni_name

uni_headers2 = list(uni_headers2.items())

uni_headers2[:5] # Test

[((1, 29), 'مخصوص داوطلبان بومي استان ايلام'),
 ((1, 41), 'مخصوص داوطلبان بومي استان بوشهر'),
 ((1, 47), 'مخصوص داوطلبان بومي استان چهارمحال و بختياري'),
 ((1, 53), 'مخصوص داوطلبان بومي استان سيستان و بلوچستان'),
 ((1, 67), 'مخصوص داوطلبان بومي استان كردستان')]

In [10]:
page_list2 = []
start_row = False

for page in range(1,3):
    content = tabula.read_pdf(PDF2, lattice = False,
                               guess = False, columns = [205, 233, 261, 290, 318, 389, 531],
                              pandas_options = {'header': None}, pages = str(page))[0]
    
    university = ''

    for idx, col in content.iterrows():

        if str(col[7]).isdigit():
            university = find_university(page, idx, uni_headers2)
            row_list = list(col.values)
            row_list += [university] 
            row_list.reverse()

            if not start_row:
                if row_list[1] == "2223":
                    start_row = True
                else:
                    continue

            if page == 1 and row_list[1] == "2237":
                row_list[8] = "دانشگاه اصفهان ـ با سپردن تعهد خدمت بـه معاونـت درمـان دانشگاه علوم پزشكي زاهدان"
            if page == 1 and row_list[1] == "2243":
                row_list[2] = "مهندسی کشاورزی-مکانیک ماشینهای کشاورزی"
            if page == 1 and row_list[1] == "7074":
                row_list[0] = "مخصوص داوطلبان بومي استان سيستان و بلوچستان"
            if page == 1 and row_list[1] == "2249":
                row_list[0] = "مخصوص داوطلبان بومي استان كردستان"
            if page == 2 and row_list[1] == "2281":
                row_list[3] = "الكترونيك, قدرت, كنترل, مخابرات"

            if page == 1 and idx < 40:
                print(idx, row_list) # Test
            page_list2.append(row_list)

36 ['مخصوص داوطلبان بومي استان ايلام', '2223', 'مهندسي برق', 'قدرت', '5', '-', '1', '4', 'دانشگاه ايلام']
37 ['مخصوص داوطلبان بومي استان ايلام', '2224', 'مهندسي شيمي', nan, '2', '-', '1', '1', 'دانشگاه ايلام']
38 ['مخصوص داوطلبان بومي استان ايلام', '2225', 'مهندسي عمران', nan, '2', '-', '1', '1', 'دانشگاه ايلام']
39 ['مخصوص داوطلبان بومي استان ايلام', '2226', 'مهندسي فناوري اطلاعات', nan, '2', '-', '1', '1', 'دانشگاه ايلام']


Convert extracted tables into a DataFrame with labeled columns.

In [11]:
df2 = pd.DataFrame(page_list2, columns = ['University', 'Program-Code', 'Program-Major-Name', 'Field-Type',
                           'First-Cap', 'Second-Cap', 'Female-Cap', 'Male-Cap', 'Description'])

df2.head(5) # Test

,University,Program-Code,Program-Major-Name,Field-Type,First-Cap,Second-Cap,Female-Cap,Male-Cap,Description
0,مخصوص داوطلبان بومي استان ايلام,2223,مهندسي برق,قدرت,5,-,1,4,دانشگاه ايلام
1,مخصوص داوطلبان بومي استان ايلام,2224,مهندسي شيمي,NaN,2,-,1,1,دانشگاه ايلام
2,مخصوص داوطلبان بومي استان ايلام,2225,مهندسي عمران,NaN,2,-,1,1,دانشگاه ايلام
3,مخصوص داوطلبان بومي استان ايلام,2226,مهندسي فناوري اطلاعات,NaN,2,-,1,1,دانشگاه ايلام
4,مخصوص داوطلبان بومي استان ايلام,2227,مهندسي كامپيوتر,نرم افزار,2,-,1,1,دانشگاه ايلام


Save the DataFrame as a CSV file in the **"OUT_TABLE"** folder.

In [12]:
os.chdir("../4.OUT_TABLE")
df2.to_csv("2.Mahroum.csv", index = False, encoding = "utf-8-sig")

---

Set the Directory
- Choose PDF 3: (Farhangian.pdf)

In [13]:
os.chdir("../3.OUT_PDF")
PDF3 = "3.Farhangian.pdf"

The following is the list of persian keywords for PDF file (3):

<div align="center">

`["مخصوص", "ادامه"]`

</div>

We will use the main pattern again to extract tables from this PDF file.

In [14]:
keywords3 = ["مخصوص", "ادامه"]

uni_headers3 = {}

for page in range(1,4):
    content = tabula.read_pdf(PDF3, lattice = False, guess = False,
                                  pandas_options = {'header': None}, pages = str(page))[0]

    for idx, col in content.iterrows():

        if page == 3 and idx >= 55:
            break

        if any(str(cell).startswith(tuple(keywords3)) for cell in col):
            # print( "page:", page, "index:", idx, "university name:", col[0])
            
            uni_name = str(col[0])
            uni_headers3[(page, idx)] = uni_name

uni_headers3 = list(uni_headers3.items())

uni_headers3[:5] # Test

[((1, 18), 'مخصوص داوطلبان بومي استان آذربايجان شرقي'),
 ((1, 26), 'مخصوص داوطلبان بومي استان آذربايجان غربي'),
 ((1, 31), 'مخصوص داوطلبان بومي استان اردبيل'),
 ((1, 36), 'مخصوص داوطلبان بومي استان اصفهان'),
 ((1, 42), 'مخصوص داوطلبان بومي استان البرز')]

In [15]:
page_list3 = []

for page in range(1,4):
    content = tabula.read_pdf(PDF3, lattice = False,
                               guess = False, columns = [205, 233, 261, 290, 318, 389, 531],
                              pandas_options = {'header': None}, pages = str(page))[0]
    
    university = ''

    for idx, col in content.iterrows():
        idx_prim = idx - 1 if (1 <= page <= 2) else idx
        
        if str(col[7]).isdigit():
            university = find_university(page, idx_prim, uni_headers3)
            row_list = list(col.values)
            row_list += [university] 
            row_list.reverse()

            if page == 1 and idx < 30:
                print(idx, row_list) # Test
            page_list3.append(row_list)

24 ['مخصوص داوطلبان بومي استان آذربايجان شرقي', '2312', 'دبيري رياضي', nan, '28', '-', 'ـــ', 'مرد', 'پرديس علامه اميني ـ تبريز']
25 ['مخصوص داوطلبان بومي استان آذربايجان شرقي', '2313', 'دبيري فيزيك', nan, '23', '-', 'ـــ', 'مرد', 'پرديس علامه اميني ـ  تبريز']
26 ['مخصوص داوطلبان بومي استان آذربايجان شرقي', '2314', 'دبيري رياضي', nan, '7', '-', 'زن', 'ـــ', 'پرديس علامه طباطبايي ـ اروميه']
27 ['مخصوص داوطلبان بومي استان آذربايجان شرقي', '2315', 'دبيري فيزيك', nan, '18', '-', 'زن', 'ـــ', 'پرديس فاطمه الزهرا)س(   ـ تبريز']
29 ['مخصوص داوطلبان بومي استان آذربايجان غربي', '2316', 'دبيري رياضي', nan, '9', '-', 'ـــ', 'مرد', 'پرديس شهيد رجايي ـ اروميه']


Convert extracted tables into a DataFrame with labeled columns.

In [16]:
df3 = pd.DataFrame(page_list3, columns = ['University', 'Program-Code', 'Program-Major-Name', 'Field-Type',
                           'First-Cap', 'Second-Cap', 'Female-Cap', 'Male-Cap', 'Description'])

df3.head(5) # Test

,University,Program-Code,Program-Major-Name,Field-Type,First-Cap,Second-Cap,Female-Cap,Male-Cap,Description
0,مخصوص داوطلبان بومي استان آذربايجان شرقي,2312,دبيري رياضي,NaN,28,-,ـــ,مرد,پرديس علامه اميني ـ تبريز
1,مخصوص داوطلبان بومي استان آذربايجان شرقي,2313,دبيري فيزيك,NaN,23,-,ـــ,مرد,پرديس علامه اميني ـ تبريز
2,مخصوص داوطلبان بومي استان آذربايجان شرقي,2314,دبيري رياضي,NaN,7,-,زن,ـــ,پرديس علامه طباطبايي ـ اروميه
3,مخصوص داوطلبان بومي استان آذربايجان شرقي,2315,دبيري فيزيك,NaN,18,-,زن,ـــ,پرديس فاطمه الزهرا)س( ـ تبريز
4,مخصوص داوطلبان بومي استان آذربايجان غربي,2316,دبيري رياضي,NaN,9,-,ـــ,مرد,پرديس شهيد رجايي ـ اروميه


Save the DataFrame as a CSV file in the **"OUT_TABLE"** folder.

In [17]:
os.chdir("../4.OUT_TABLE")
df3.to_csv("3.Farhangian.csv", index = False, encoding = "utf-8-sig")

---

Set the Directory
- Choose PDF 4: (Kaardani_Farhangian.pdf)

In [18]:
os.chdir("../3.OUT_PDF")
PDF4 = "4.Kaardani_Farhangian.pdf"

**Important Note:** This PDF does not have any **header**, making it easy to iterate through the rows.

The following list contains the coordinates of the vertical lines. (from Left to Right)

- The area of this table has been modified. 

<div align="center">

`[236, 264, 293, 321, 349, 491, 522]`

</div>

The following image shows an example of how the rows is detected (highlighted in red & blue).

<div align="center">
  <img src="image/3.ROW_Farhangian.png" width="700"/>
</div>

In [19]:
page_list4 = []

for page in range(1,3):
    content = tabula.read_pdf(PDF4, lattice = False,
                               guess = False, columns = [236, 264, 293, 321, 349, 491, 522],
                              pandas_options = {'header': None}, pages = str(page))[0]

    for idx, col in content.iterrows():

        if str(col[6]).isdigit():

            row_list = list(col.values)
            row_list.reverse()
            row_list[2] = 'مخصوص داوطلبان بومی استان ' + row_list[2]
            row_list[0] = 'کاردانی آموزش و پرورش ابتدایی'
            row_list.append(None)
            order = [2,1,0,8,3,4,5,6,7]
            row_list = [row_list[i] for i in order]

            if page == 1 and idx < 62:
                print(row_list) # Test
            page_list4.append(row_list)

['مخصوص داوطلبان بومی استان آذربايجان شرقي', '7087', 'کاردانی آموزش و پرورش ابتدایی', None, '30', '-', 'ـــ', 'مرد', 'پرديس علامه اميني ـ  تبريز']
['مخصوص داوطلبان بومی استان آذربايجان شرقي', '7088', 'کاردانی آموزش و پرورش ابتدایی', None, '30', '-', 'زن', 'ـــ', 'پرديس فاطمه الزهرا)س(   ـ تبريز']
['مخصوص داوطلبان بومی استان آذربايجان غربي', '7089', 'کاردانی آموزش و پرورش ابتدایی', None, '30', '-', 'ـــ', 'مرد', 'پرديس شهيد رجايي ـ اروميه']
['مخصوص داوطلبان بومی استان آذربايجان غربي', '7090', 'کاردانی آموزش و پرورش ابتدایی', None, '30', '-', 'زن', 'ـــ', 'پرديس علامه طباطبايي ـ اروميه']
['مخصوص داوطلبان بومی استان اردبيل', '7091', 'کاردانی آموزش و پرورش ابتدایی', None, '30', '-', 'ـــ', 'مرد', 'پرديس علامه طباطبايي ـ اردبيل']


Convert extracted tables into a DataFrame with labeled columns.

In [20]:
df4 = pd.DataFrame(page_list4, columns = ['University', 'Program-Code', 'Program-Major-Name', 'Field-Type',
                               'First-Cap', 'Second-Cap', 'Female-Cap', 'Male-Cap', 'Description'])

df4.head(5) # Test

,University,Program-Code,Program-Major-Name,Field-Type,First-Cap,Second-Cap,Female-Cap,Male-Cap,Description
0,مخصوص داوطلبان بومی استان آذربايجان شرقي,7087,کاردانی آموزش و پرورش ابتدایی,None,30,-,ـــ,مرد,پرديس علامه اميني ـ تبريز
1,مخصوص داوطلبان بومی استان آذربايجان شرقي,7088,کاردانی آموزش و پرورش ابتدایی,None,30,-,زن,ـــ,پرديس فاطمه الزهرا)س( ـ تبريز
2,مخصوص داوطلبان بومی استان آذربايجان غربي,7089,کاردانی آموزش و پرورش ابتدایی,None,30,-,ـــ,مرد,پرديس شهيد رجايي ـ اروميه
3,مخصوص داوطلبان بومی استان آذربايجان غربي,7090,کاردانی آموزش و پرورش ابتدایی,None,30,-,زن,ـــ,پرديس علامه طباطبايي ـ اروميه
4,مخصوص داوطلبان بومی استان اردبيل,7091,کاردانی آموزش و پرورش ابتدایی,None,30,-,ـــ,مرد,پرديس علامه طباطبايي ـ اردبيل


Save this file as a CSV in the **"OUT_TABLE"** folder.

In [21]:
os.chdir("../4.OUT_TABLE")
df4.to_csv("4.Kaardani_Farhangian.csv", index = False, encoding = "utf-8-sig")

---

Set the Directory
- Choose PDF 5: (Nimeh_Motemarkez.pdf)

In [22]:
os.chdir("../3.OUT_PDF")
PDF5 = "5.Nimeh_Motemarkez.pdf"

The following is the list of persian keywords for PDF file (5):

<div align="center">

`["دانشكده", "مركز", "دانشگاه"]`

</div>

**Important Note:** This PDF is a bit **challenging** because it contains three different types of tables.

The following lists contain the coordinates of the vertical lines. (from Left to Right)

- The area of each table has been modified. 

<div align="center">

`[204, 233, 261, 289, 317, 389, 530]`

`[232, 259, 286, 313, 340, 439, 530]`

`[236, 265, 293, 322, 350, 492, 522]`

</div>

The following image shows an example of how the rows is detected (highlighted in red & blue).

<div align="center">
  <img src="image/4.ROW_Nime_Motemarkez.png" width="700"/>
</div>

In [23]:
keywords5 = ["دانشكده", "مركز", "دانشگاه"]

uni_headers5 = {}

for page in range(1,2):
    content = tabula.read_pdf(PDF5, lattice = False, guess = False,
                                  pandas_options = {'header': None}, pages = str(page))[0]

    for idx, col in content.iterrows():

        if idx <= 24:
            continue

        if any(str(cell).startswith(tuple(keywords5)) for cell in col):
            # print( "page:", page, "index:", idx, "university name:", col[0])
            
            uni_name = str(col[0])
            uni_headers5[(page, idx)] = uni_name

uni_headers5 = list(uni_headers5.items())

uni_headers5[:5] # Test

[((1, 40), 'دانشگاه تهران'),
 ((1, 47), 'دانشگاه شهيد باهنركرمان'),
 ((1, 49), 'دانشگاه صنعت نفت'),
 ((1, 51), 'مركزتحصيلات تكميلي در علوم پايه زنجان'),
 ((1, 53), 'دانشكده اطلاعات وابسته به وزارت اطلاعات')]

In [24]:
page_list5 = []

for page in range(1,4):

    if page == 1:

        content = tabula.read_pdf(PDF5, lattice = False,
                               guess = False, columns = [204, 233, 261, 289, 317, 389, 530],
                              pandas_options = {'header': None}, pages = str(page))[0]
        
        university = ''

        for idx, col in content.iterrows():

            if str(col[7]).isdigit():
                university = find_university(page, idx, uni_headers5)
                row_list = list(col.values)
                row_list += [university] 
                row_list.reverse()

                if idx < 55:
                    print(idx, row_list) # Test
                page_list5.append(row_list)
    
    if page == 2:

        content = tabula.read_pdf(PDF5, lattice = False,
                               guess = False, columns = [232, 259, 286, 313, 340, 439, 530],
                              pandas_options = {'header': None}, pages = str(page))[0]

        start_row = False

        for idx, col in content.iterrows():
            row_list = list(col.values)
            row_list.reverse()
            row_list.append(None)

            if not start_row:
                if row_list[1] == "تربيت بدني و علوم ورزشي":
                    start_row = True
                else:
                    continue

            if str(row_list[0]) == "7020":
                break

            if row_list[7] == "دانشگاه تربيت دبيرشهيد رجايي تهران **":
                row_list[0] = "7020"
                row_list[1] = "تربيت بدني و علوم ورزشي"
                row_list[2] = None
                row_list[3] = "-"
                row_list[4] = "40"
                row_list[5] = "15"
                row_list[6] = "25"
                row_list[8] = "ویژه داوطلبان بومی استان تهران"
            
            if pd.isna(row_list[0]):
                row_list[0] = "7004"

            if row_list[7] == "دانشگاه بوعلي سينا همدان ـ محل تحصيل دانشكده نهاوند- فاقد خوابگاه":
                row_list[7] = "دانشگاه بوعلي سينا همدان"
                row_list[8] = "محل تحصیل دانشکده نهاوند-فاقد خوابگاه"

            if row_list[7] == "دانشگاه خوارزمي تهران ـ محل تحصيل كرج)فاقد خوابگاه(":
                row_list[7] = "دانشگاه خوارزمی تهران"
                row_list[8] = "محل تحصیل کرج (فاقد خوابگاه)"

            if row_list[7] == "دانشگاه خوارزمي تهران ـ محل تحصيل كرج":
                row_list[7] = "دانشگاه خوارزمی تهران"
                row_list[8] = "محل تحصیل کرج"

            if row_list[7] == "دانشگاه گيلان -رشت ـ عنوان رشته مربيگري ورزشي":
                row_list[7] = "دانشگاه گیلان-رشت"
                row_list[8] = "عنوان رشته مربیگری ورزشی"
            
            order = [7,0,1,2,3,4,5,6,8]
            row_list = [row_list[i] for i in order]

            # print(idx, row_list) # Test
            page_list5.append(row_list)

    if page == 3:

        content = tabula.read_pdf(PDF5, lattice = False,
                               guess = False, columns = [236, 265, 293, 322, 350, 492, 522],
                              pandas_options = {'header': None}, pages = str(page))[0]
        
        for idx, col in content.iterrows():

            if str(col[6]).isdigit():

                row_list = list(col.values)
                row_list.reverse()
                row_list[2] = 'مخصوص داوطلبان بومی استان ' + row_list[2]
                row_list[0] = 'تربیت بدنی و علوم ورزشی'
                row_list.append(None)
                order = [2,1,0,8,3,4,5,6,7]
                row_list = [row_list[i] for i in order]

                # print(idx, row_list) # Test
                page_list5.append(row_list)

46 ['دانشگاه تهران', '7021', 'دكتراي پيوسته بيوتكنولوژي', nan, '-', '8', 'زن', 'مرد', nan]
48 ['دانشگاه شهيد باهنركرمان', '2219', 'دكتراي پيوسته رياضي', nan, '-', '10', '5', '5', 'پذيرش ازرتبه هاي كمترازهزار']
50 ['دانشگاه صنعت نفت', '2220', 'مهندسي كشتي', 'موتور', '-', '25', 'ـــ', 'مرد', 'محل تحصيل محمودآباد مازندران']
52 ['مركزتحصيلات تكميلي در علوم پايه زنجان', '2221', 'دكتراي پيوسته فيزيك', nan, '-', '20', 'زن', 'مرد', 'داراي امكانات خوابگاهي']
54 ['دانشكده اطلاعات وابسته به وزارت اطلاعات', '7061', 'علوم اجتماعي', 'مطالعات امنيتي', '-', '100', 'ـــ', 'مرد', nan]


Convert extracted tables into a DataFrame with labeled columns.

In [25]:
df5 = pd.DataFrame(page_list5, columns = ['University', 'Program-Code', 'Program-Major-Name', 'Field-Type',
                               'First-Cap', 'Second-Cap', 'Female-Cap', 'Male-Cap', 'Description'])

df5.head(5) # Test

,University,Program-Code,Program-Major-Name,Field-Type,First-Cap,Second-Cap,Female-Cap,Male-Cap,Description
0,دانشگاه تهران,7021,دكتراي پيوسته بيوتكنولوژي,NaN,-,8,زن,مرد,NaN
1,دانشگاه شهيد باهنركرمان,2219,دكتراي پيوسته رياضي,NaN,-,10,5,5,پذيرش ازرتبه هاي كمترازهزار
2,دانشگاه صنعت نفت,2220,مهندسي كشتي,موتور,-,25,ـــ,مرد,محل تحصيل محمودآباد مازندران
3,مركزتحصيلات تكميلي در علوم پايه زنجان,2221,دكتراي پيوسته فيزيك,NaN,-,20,زن,مرد,داراي امكانات خوابگاهي
4,دانشكده اطلاعات وابسته به وزارت اطلاعات,7061,علوم اجتماعي,مطالعات امنيتي,-,100,ـــ,مرد,NaN


Save this file as a CSV in the **"OUT_TABLE"** folder.

In [26]:
os.chdir("../4.OUT_TABLE")
df5.to_csv("5.Nimeh_Motemarkez.csv", index = False, encoding = "utf-8-sig")

---

Set the Directory
- Choose PDF 6: (Shabaneh.pdf)

In [27]:
os.chdir("../3.OUT_PDF")
PDF6 = "6.Shabaneh.pdf"

The following is the list of persian keywords for PDF file (6):

<div align="center">

`["دانشگاه", "آموزشكده", "دانشكده", "مجتمع", "مركز", "ادامه", "اموزشكده"]`

</div>

We will use the main pattern again to extract tables from this PDF file.

In [28]:
keywords6 = ["دانشگاه", "آموزشكده", "دانشكده", "مجتمع", "مركز", "ادامه", "اموزشكده"]

uni_headers6 = {}

for page in range(1,12):
    content = tabula.read_pdf(PDF6, lattice = False, guess = False,
                                  pandas_options = {'header': None}, pages = str(page))[0]

    for idx, col in content.iterrows():

        if any(str(cell).startswith(tuple(keywords6)) for cell in col):
            # print( "page:", page, "index:", idx, "university name:", col[0])
            
            uni_name = str(col[0])
            uni_headers6[(page, idx)] = uni_name

uni_headers6 = list(uni_headers6.items())

uni_headers6[:5] # Test

[((1, 12), 'دانشگاه آيت اله بروجردي- بروجرد'),
 ((1, 24), 'دانشگاه اراك'),
 ((1, 46), 'دانشگاه اروميه'),
 ((1, 72), 'دانشگاه الزهرا)س-( تهران'),
 ((2, 1), 'ادامه دانشگاه الزهرا)س-( تهران')]

In [29]:
page_list6 = []

for page in range(1,12):
    content = tabula.read_pdf(PDF6, lattice = False,
                               guess = False, columns = [205, 233, 261, 290, 318, 389, 531],
                              pandas_options = {'header': None}, pages = str(page))[0]
    university = ''

    for idx, col in content.iterrows():
        
        if str(col[7]).isdigit():
            university = find_university(page, idx, uni_headers6)
            row_list = list(col.values)
            row_list += [university] 
            row_list.reverse()

            if page == 1 and row_list[1] == "2442" or row_list[1] == "2448" or row_list[1] == "2451":
                row_list[8] = "محل تحصيل مركزآموزش عالي محلات-فاقـدخوابگاه وامكانـات رفاهي"
            if page == 3 and row_list[1] == "2536":
                row_list[3] = "اتمی، حالت جامد، مولکولی"
            if page == 5 and row_list[1] == "2652":
                row_list[3] = "الكترونيك, قدرت, كنترل, مخابرات"
            if page == 8 and row_list[1] == "2812":
                row_list[3] = "الكترونيك, قدرت, كنترل, مخابرات"
            if page == 10 and row_list[1] == "2941":
                row_list[2] = "کاردانی علمی کاربردی ماشینهای کشاورزی-مکانیزاسیون ماشینهای کشاورزی"
            if page == 11 and row_list[1] == "7330":
                row_list[8] = "كــارداني مــديريت صــنعت جهــانگردي-گــرايش مــديريت جهانگردي"

            if page == 1 and idx < 23:
                print(idx, row_list) # Test
            page_list6.append(row_list)

18 ['دانشگاه آيت اله بروجردي- بروجرد', '2436', 'رياضيات وكاربردها', nan, '-', '45', 'زن', 'مرد', nan]
19 ['دانشگاه آيت اله بروجردي- بروجرد', '2437', 'فيزيك', nan, '-', '45', 'زن', 'مرد', nan]
20 ['دانشگاه آيت اله بروجردي- بروجرد', '2438', 'مهندسي عمران', nan, '-', '45', 'زن', 'مرد', nan]
21 ['دانشگاه آيت اله بروجردي- بروجرد', '2439', 'مهندسي كامپيوتر', 'نرم افزار', '-', '45', 'زن', 'مرد', nan]
22 ['دانشگاه آيت اله بروجردي- بروجرد', '7184', 'علوم اقتصادي', 'اقتصادبازرگاني', '-', '45', 'زن', 'مرد', nan]


Convert extracted tables into a DataFrame with labeled columns.

In [30]:
df6 = pd.DataFrame(page_list6, columns = ['University', 'Program-Code', 'Program-Major-Name', 'Field-Type',
                               'First-Cap', 'Second-Cap', 'Female-Cap', 'Male-Cap', 'Description'])

df6.head(5) # Test

,University,Program-Code,Program-Major-Name,Field-Type,First-Cap,Second-Cap,Female-Cap,Male-Cap,Description
0,دانشگاه آيت اله بروجردي- بروجرد,2436,رياضيات وكاربردها,NaN,-,45,زن,مرد,NaN
1,دانشگاه آيت اله بروجردي- بروجرد,2437,فيزيك,NaN,-,45,زن,مرد,NaN
2,دانشگاه آيت اله بروجردي- بروجرد,2438,مهندسي عمران,NaN,-,45,زن,مرد,NaN
3,دانشگاه آيت اله بروجردي- بروجرد,2439,مهندسي كامپيوتر,نرم افزار,-,45,زن,مرد,NaN
4,دانشگاه آيت اله بروجردي- بروجرد,7184,علوم اقتصادي,اقتصادبازرگاني,-,45,زن,مرد,NaN


Save this file as a CSV in the **"OUT_TABLE"** folder.

In [31]:
os.chdir("../4.OUT_TABLE")
df6.to_csv("6.Shabaneh.csv", index = False, encoding = "utf-8-sig")

---

Set the Directory
- Choose PDF 7: (Nimeh_Hozouri.pdf)

In [32]:
os.chdir("../3.OUT_PDF")
PDF7 = "7.Nimeh_Hozouri.pdf"

The following is the list of persian keywords for PDF file (7):

<div align="center">

`["دانشگاه", "موسسه"]`

</div>

**Important Note:** This PDF is also a bit **challenging** because it contains 2 different types of tables.

- One page does not include a header, while the following page contains a header.

The following image shows an example of how the rows is detected (highlighted in red & blue).

<div align="center">
  <img src="image/5.ROW_Nime_Hozouri.png" width="700"/>
</div>

In [33]:
keywords7 = ["دانشگاه", "موسسه"]

uni_headers7 = {}

for page in range(2,3):
    content = tabula.read_pdf(PDF7, lattice = False, guess = False,
                                  pandas_options = {'header': None}, pages = str(page))[0]

    for idx, col in content.iterrows():

        if any(str(cell).startswith(tuple(keywords7)) for cell in col):
            # print( "page:", page, "index:", idx, "university name:", col[0])

            uni_name = str(col[0])
            uni_headers7[(page, idx)] = uni_name

uni_headers7 = list(uni_headers7.items())

uni_headers7[:5] # Test

[((2, 12), 'دانشگاه اروميه'),
 ((2, 21), 'دانشگاه علامه طباطبايي - تهران'),
 ((2, 23), 'دانشگاه كردستان - سنندج'),
 ((2, 25), 'موسسه اموزشي وپژوهشي امام خميني)ره-(  قم*')]

In [34]:
page_list7 = []

for page in range(1,3):
    content = tabula.read_pdf(PDF7, lattice = False,
                               guess = False, columns = [204, 233, 261, 289, 317, 389, 530],
                              pandas_options = {'header': None}, pages = str(page))[0]
    if page == 1:

        for idx, col in content.iterrows():

            if str(col[6]) == "تربيت بدني و علوم ورزشي":
                row_list = list(col.values)
                row_list.reverse()
                row_list[0] = "7333"
                row_list.append(None)
                if pd.isna(row_list[7]):
                    row_list[7] = "دانشگاه بوعلي سينا همدان"
                    row_list[8] = "محل تحصیل دانشکده نهاوند-فاقد خوابگاه"

                if row_list[7] == "دانشگاه جهرم":
                    row_list[2] = "فيزيولــــوژي ورزشــــي, مديريت ورزشي"

                if row_list[7] == "دانشگاه گيلان - رشت ـ  عنوان رشته مربيگري ورزشي":
                    row_list[7] = "دانشگاه گيلان - رشت"
                    row_list[8] = "عنوان رشته مربیگری ورزشی"

                order = [7,0,1,2,3,4,5,6,8]
                row_list = [row_list[i] for i in order]

                if idx < 27:
                    print(idx, row_list) # Test
                page_list7.append(row_list)
    
    else:

        university = ''

        for idx, col in content.iterrows():

            if str(col[7]).isdigit():
                university = find_university(page, idx, uni_headers7)
                row_list = list(col.values)
                row_list += [university] 
                row_list.reverse()

                # print(idx, row_list) # Test
                page_list7.append(row_list)

22 ['دانشگاه اراك', '7333', 'تربيت بدني و علوم ورزشي', nan, '-', '20', 'ـــ', 'مرد', None]
23 ['دانشگاه اروميه', '7333', 'تربيت بدني و علوم ورزشي', nan, '-', '5', 'زن', 'مرد', None]
24 ['دانشگاه الزهرا)س-(تهران', '7333', 'تربيت بدني و علوم ورزشي', 'رفتارحركتي', '-', '12', 'زن', 'ـــ', None]
25 ['دانشگاه الزهرا)س-(تهران', '7333', 'تربيت بدني و علوم ورزشي', 'فيزيولوژي ورزشي', '-', '12', 'زن', 'ـــ', None]
26 ['دانشگاه الزهرا)س-(تهران', '7333', 'تربيت بدني و علوم ورزشي', 'مديريت ورزشي', '-', '12', 'زن', 'ـــ', None]


Convert extracted tables into a DataFrame with labeled columns.

In [35]:
df7 = pd.DataFrame(page_list7, columns = ['University', 'Program-Code', 'Program-Major-Name', 'Field-Type',
                               'First-Cap', 'Second-Cap', 'Female-Cap', 'Male-Cap', 'Description'])

df7.head(5) # Test

,University,Program-Code,Program-Major-Name,Field-Type,First-Cap,Second-Cap,Female-Cap,Male-Cap,Description
0,دانشگاه اراك,7333,تربيت بدني و علوم ورزشي,NaN,-,20,ـــ,مرد,None
1,دانشگاه اروميه,7333,تربيت بدني و علوم ورزشي,NaN,-,5,زن,مرد,None
2,دانشگاه الزهرا)س-(تهران,7333,تربيت بدني و علوم ورزشي,رفتارحركتي,-,12,زن,ـــ,None
3,دانشگاه الزهرا)س-(تهران,7333,تربيت بدني و علوم ورزشي,فيزيولوژي ورزشي,-,12,زن,ـــ,None
4,دانشگاه الزهرا)س-(تهران,7333,تربيت بدني و علوم ورزشي,مديريت ورزشي,-,12,زن,ـــ,None


Save this file as a CSV in the **"OUT_TABLE"** folder.

In [36]:
os.chdir("../4.OUT_TABLE")
df7.to_csv("7.Nimeh_Hozouri.csv", index = False, encoding = "utf-8-sig")

---

Set the Directory
- Choose PDF 8: (Payam_Nour.pdf)

In [37]:
# os.chdir("../3.OUT_PDF")
# PDF8 = "8.Payam_Nour.pdf"

In [38]:
# keywords8 = ["دانشگاه"]

# uni_headers8 = {}

# for page in range(1,2):

#     left_content = tabula.read_pdf(PDF8, guess=False, lattice=False, area=[0, 0, 795, 298],
#                                    pandas_options={'header': None}, pages=str(page))[0]

#     right_content = tabula.read_pdf(PDF8, guess=False, lattice=False, area=[0, 298, 795, 576],
#                                    pandas_options={'header': None}, pages=str(page))[0]


#     for idx, col in left_content.iterrows():
#         if any(str(cell).startswith(tuple(keywords8)) for cell in col):
#                 print( "page:", page, "index:", idx, "university name:", col[0])

---

Set the Directory
- Choose PDF 9: (Gheir_Entefai.pdf)

In [39]:
os.chdir("../3.OUT_PDF")
PDF9 = "9.Gheir_Entefai.pdf"

The following is the list of persian keywords for PDF file (9):

<div align="center">

`["دانشكده", "دانشگاه", "ادامه", "موسسات", "موسسه"]`

</div>

We will use the main pattern again to extract tables from this PDF file.

In [40]:
keywords9 = ["دانشكده", "دانشگاه", "ادامه", "موسسات", "موسسه"]

uni_headers9 = {}

for page in range(1,21):
    content = tabula.read_pdf(PDF9, lattice = False, guess = False,
                                  pandas_options = {'header': None}, pages = str(page))[0]

    for idx, col in content.iterrows():

        if any(str(cell).startswith(tuple(keywords9)) for cell in col):
            # print( "page:", page, "index:", idx, "university name:", col[0])

            uni_name = str(col[0])
            uni_headers9[(page, idx)] = uni_name

uni_headers9 = list(uni_headers9.items())

uni_headers9[:5] # Test

[((1, 8), 'دانشكده آمار و انفورماتيك سازمان سنجش آموزش كشور)غيرانتفاعي('),
 ((1, 16), 'دانشكده اصول الدين )غيرانتفاعي(*'),
 ((1, 38), 'دانشكده معارف قرآني- اصفهان)غيرانتفاعي('),
 ((1, 47), 'دانشگاه غير انتفاعي بين المللي امام رضا )ع('),
 ((1, 60), 'دانشگاه غيرانتفاعي شمال -  آمل')]

**Important Note:** My approach does not recognize the headers on pages 3 and 10 in this PDF file, and I could not find out why!
- As a basic solution, I added them manually by defining lists for those specific pages.

In [ ]:
uni_range_page3 = [
    ((7, 9),  "موسسه غيرانتفاعي اثيرالدين ابهري- ابهر"),
    ((11, 14), "موسسه غیرانتفاعی احرار- رشت"),
    ((16, 24), "موسسه غيرانتفاعي اديب مازندران - ساري"),
    ((26, 31), "موسسه غيرانتفاعي اديبان - گرمسار"),
    ((33, 37), "موسسه غيرانتفاعي آذرابادگان - اروميه"),
    ((39,40), "موسسه غيرانتفاعي ارس- تبريز"),
    ((41,52), "موسسه غيرانتفاعي ارشاد- دماوند"),
    ((54,55), "موسسه غيرانتفاعي ارم- شيراز"),
    ((57,58), "موسسه غيرانتفاعي استرآباد- گرگان"),
    ((59,62), "موسسه غيرانتفاعي اسرار- مشهد"),
    ((64,67), "موسسه غيرانتفاعي اسوه -تبريز"),
    ((69,74), "موسسه غيرانتفاعي اشراق- بجنورد"),
    ((76,78), "ادامه موسسه غيرانتفاعي اشراق- بجنورد"),
    ((80,81), "موسسه غيرانتفاعي آفرينش- بروجرد"),]

uni_range_page10 = [
    ((7,13), "موسسه غيرانتفاعي ديلمان- لاهيجان"),
    ((15,21), "موسسه غیرانتفاعی راغب اصفهانی- اصفهان"),
    ((23,29), "موسسه غيرانتفاعي راه دانش - بابل"),
    ((31,35), "موسسه غيرانتفاعي راهبردشمال - رشت"),
    ((37,39), "موسسه غيرانتفاعي راهيان نور- ساري"),
    ((41,42), "موسسه غيرانتفاعي رايان دانش- قائمشهر"),
    ((43,44), "موسسه غيرانتفاعي ربع رشيدي - تبريز"),
    ((46,55), "موسسه غيرانتفاعي رجا- قزوين"),
    ((57,60), "موسسه غيرانتفاعي رحمان - رامسر"),
    ((62,63), "موسسه غيرانتفاعي رسام - كرج"),
    ((64,68), "موسسه غيرانتفاعي رشددانش - سمنان"),
    ((70,76), "موسسه غيرانتفاعي رشديه - تبريز"),
    ((78,81), "موسسه غيرانتفاعي رودكي - تنكابن"),]

row_page20 = [
    ["دانشگاه غيرانتفاعي شمال -امل", "9897", "تربيت بدني و علوم ورزشي", None, "-", "60", "زن", "مرد", None],
    ["موسسه غيرانتفاعي اذرابادگان -اروميه", "9897", "تربيت بدني و علوم ورزشي", None, "-", "60", "زن", "مرد", None],
    ["موسسه غيرانتفاعي ايوانكي", "9897", "تربيت بدني و علوم ورزشي", "مـــديريت ورزشـــي, مربيگري ورزشي", "-", "60", "زن", "مرد", None],
    ["موسسه غيرانتفاعي شفق -تنكابن", "9897", "تربيت بدني و علوم ورزشي", "مديريت ورزشي", "-", "60", "زن", "مرد", None],]

page_list9 = []

for page in range(1,21):
    content = tabula.read_pdf(PDF9, lattice = False,
                               guess = False, columns = [205, 233, 261, 290, 318, 389, 531],
                              pandas_options = {'header': None}, pages = str(page))[0]
    university = ''

    for idx, col in content.iterrows():

        if str(col[7]).isdigit():
            idx_prim = idx - 1 if (page == 2 or 5 <= page <= 8 or 12 <= page <= 16 or page == 18 or page == 20) else idx
            university = find_university(page, idx_prim, uni_headers9)
            row_list = list(col.values)
            row_list += [university] 
            row_list.reverse()

            if page == 1 and row_list[0] == "دانشگاه غير انتفاعي بين المللي امام رضا )ع(":
                row_list[8] = "محل تحصيل خواهران و برادران در دو پرديس جداگانه ميباشد."

            if page == 3:
                for (start_idx, end_idx), name in uni_range_page3:
                    if start_idx <= idx <= end_idx:
                        row_list[0] = name
                        break
            
                if idx == 14:
                    row_list[3] = "تاسیسات ابرسانی و گازرسانی"
            
            if page == 4 and row_list[1] == "4354":
                row_list[3] = "تاسیسات بهداشتی و گازرسانی"

            if page == 5 and row_list[1] == "4361":
                row_list[2] = "کاردانی علمی کاربردی بهینه سازی مصرف انرژی-صنعت"

            if page == 5 and row_list[1] == "4404":
                row_list[2] = "کاردانی علمی کاربردی ارتباطات و فناوری اطلاعات (ICT)"

            if page == 6 and row_list[1] == "4449":
                row_list[2] = "کاردانی علمی کاربردی ارتباطات و فناوری اطلاعات (ICT)"

            if page == 7 and row_list[1] == "4478":
                row_list[2] = "کاردانی علمی کاربردی ارتباطات و فناوری اطلاعات (ICT)"

            if page == 7 and row_list[1] == "4481":
                row_list[3] = "تاسیسات بهداشتی و گازرسانی" 

            if page == 8 and row_list[1] == "4509":
                row_list[2] = "کاردانی علمی کاربردی ارتباطات و فناوری اطلاعات (ICT)"

            if page == 10:
                for (start_idx, end_idx), name in uni_range_page10:
                    if start_idx <= idx <= end_idx:
                        row_list[0] = name
                        break

            if page == 15 and row_list[1] == "4801":
                row_list[3] = "تاسیسات ابرسانی و گازرسانی"

            if page == 15 and row_list[1] == "4809":
                row_list[3] = "تاسیسات بهداشتی و گازرسانی"

            if page == 16 and row_list[1] == "4838":
                row_list[3] = "الكترونيك, قدرت, كنترل, سيستمهاي ديجيتال"

            if page == 19 and row_list[1] == "4932":
                row_list[2] = "کاردانی علمی کاربردی ماشینهای کشاورزی- مکانیزاسیون ماشینهای کشاورزی"

            if page == 20 and row_list[1] == "9897":
                page_list9.extend(row_page20)
                continue

            if page == 1 and idx < 20:
                print(idx, row_list) # Test
            page_list9.append(row_list)

14 ['دانشكده آمار و انفورماتيك سازمان سنجش آموزش كشور)غيرانتفاعي(', '4220', 'كارداني علمي -كاربردي نرم افزاركامپيوتر', nan, '100', '-', 'زن', 'مرد', nan]
15 ['دانشكده آمار و انفورماتيك سازمان سنجش آموزش كشور)غيرانتفاعي(', '4221', 'آمارو سنجش اموزشي', nan, '60', '-', 'زن', 'مرد', nan]
17 ['دانشكده اصول الدين )غيرانتفاعي(*', '9429', 'زبان وادبيات عربي', nan, '30', '-', 'زن', 'ـــ', 'محل تحصيل تهران']
18 ['دانشكده اصول الدين )غيرانتفاعي(*', '9430', 'زبان وادبيات عربي', nan, '30', '-', 'ـــ', 'مرد', 'محل تحصيل دزفول']
19 ['دانشكده اصول الدين )غيرانتفاعي(*', '9431', 'زبان وادبيات عربي', nan, '30', '-', 'زن', 'ـــ', 'محل تحصيل دزفول']


Convert extracted tables into a DataFrame with labeled columns.

In [ ]:
df9 = pd.DataFrame(page_list9, columns = ['University', 'Program-Code', 'Program-Major-Name', 'Field-Type',
                               'First-Cap', 'Second-Cap', 'Female-Cap', 'Male-Cap', 'Description'])

df9.head(5) # Test

Save this file as a CSV in the **"OUT_TABLE"** folder.

In [ ]:
os.chdir("../4.OUT_TABLE")
df9.to_csv("9.Gheir_Entefai.csv", index = False, encoding = "utf-8-sig")

---

Set the Directory
- Choose PDF 10: (Majazi.pdf)

In [ ]:
os.chdir("../3.OUT_PDF")
PDF10 = "10.Majazi.pdf"

The following is the list of persian keywords for PDF file (10):

<div align="center">

`["دانشگاه", "موسسه"]`

</div>

We will use the main pattern again to extract tables from this PDF file.

In [ ]:
keywords10 = ["دانشگاه", "موسسه"]

uni_headers10 = {}

for page in range(1,2):
    content = tabula.read_pdf(PDF10, lattice = False, guess = False,
                                  pandas_options = {'header': None}, pages = str(page))[0]

    for idx, col in content.iterrows():

        if any(str(cell).startswith(tuple(keywords10)) for cell in col):
            # print( "page:", page, "index:", idx, "university name:", col[0])

            uni_name = str(col[0])
            uni_headers10[(page, idx)] = uni_name

uni_headers10 = list(uni_headers10.items())

uni_headers10[:5] # Test

In [ ]:
page_list10 = []

for page in range(1,2):
    content = tabula.read_pdf(PDF10, lattice = False,
                               guess = False, columns = [205, 233, 261, 290, 318, 389, 531],
                              pandas_options = {'header': None}, pages = str(page))[0]
    university = ''

    for idx, col in content.iterrows():
        
        if str(col[7]).isdigit():
            university = find_university(page, idx, uni_headers10)
            row_list = list(col.values)
            row_list += [university] 
            row_list.reverse()

            if row_list[0] == "دانشگاه شيراز":
                row_list[8] = "محل تحصيل دانشكده آموزشهاي الكترونيكي"
            if row_list[0] == "دانشگاه صنعتي شريف-  تهران":
                row_list[8] = "پذيرش ازطريق پرديس بين الملل دانشگاه صنعتي شريف در جزيره كيش- محل تحصيل تهران"

            if idx < 20:
                print(idx, row_list) # Test
            page_list10.append(row_list)

Convert extracted tables into a DataFrame with labeled columns.

In [ ]:
df10 = pd.DataFrame(page_list10, columns = ['University', 'Program-Code', 'Program-Major-Name', 'Field-Type',
                               'First-Cap', 'Second-Cap', 'Female-Cap', 'Male-Cap', 'Description'])

df10.head(5) # Test

Save this file as a CSV in the **"OUT_TABLE"** folder.

In [ ]:
os.chdir("../4.OUT_TABLE")
df10.to_csv("10.Majazi.csv", index = False, encoding = "utf-8-sig")